# Análise Financeira

## Objetivo

Este notebook consolida as informações financeiras da **Distribuidora Horizonte** para acompanhamento da carteira de contas a receber e contas a pagar.

A análise permite responder perguntas como:

- Quanto a empresa possui a receber?
- Quanto está vencido?
- Qual é a taxa de inadimplência da carteira aberta?
- Qual é o prazo médio de recebimento?
- Quais clientes possuem maior exposição financeira?
- Qual é a distribuição da carteira vencida por faixa de atraso?
- Quanto a empresa possui a pagar?
- Quais são as entradas e saídas financeiras previstas?
- Qual é o saldo financeiro projetado?

## Indicadores

### Contas a receber

- saldo em aberto;
- valor a vencer;
- valor vencido;
- taxa de inadimplência;
- prazo médio de recebimento;
- aging da carteira;
- concentração da inadimplência por cliente.

### Contas a pagar

- saldo em aberto;
- títulos a vencer;
- títulos vencidos;
- compromissos financeiros futuros.

### Fluxo de caixa

- recebimentos realizados;
- pagamentos realizados;
- entradas previstas;
- saídas previstas;
- saldo financeiro mensal.

## Saídas

Este notebook cria:

- `resumo_financeiro_atual`;
- `aging_contas_receber`;
- `aging_contas_pagar`;
- `inadimplencia_clientes`;
- `fluxo_caixa_mensal`.

## Fluxo

Silver Financeiro → Indicadores → Gold

In [0]:
from datetime import timedelta

from pyspark.sql import functions as F


# ---------------------------------------------------------
# CONFIGURAÇÃO
# ---------------------------------------------------------

catalogo_atual = spark.sql(
    "SELECT current_catalog()"
).first()[0]

schema_silver = "varejo_silver"
schema_gold = "varejo_gold"


print(f"Catálogo: {catalogo_atual}")
print(f"Silver: {schema_silver}")
print(f"Gold: {schema_gold}")

In [0]:
def carregar_silver(nome_tabela):
    """
    Carrega uma tabela da camada Silver.
    """

    return spark.table(
        f"{catalogo_atual}."
        f"{schema_silver}."
        f"{nome_tabela}"
    )


def salvar_gold(
    dataframe,
    nome_tabela
):
    """
    Salva um DataFrame como tabela Delta
    na camada Gold.
    """

    nome_completo = (
        f"{catalogo_atual}."
        f"{schema_gold}."
        f"{nome_tabela}"
    )

    (
        dataframe
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(
            nome_completo
        )
    )

    print(
        f"Tabela criada: {nome_completo}"
    )

In [0]:
df_resultado_qualidade = carregar_silver(
    "resultado_qualidade"
)


testes_criticos_reprovados = (

    df_resultado_qualidade

    .filter(
        (F.col("status") == "REPROVADO")
        &
        (F.col("criticidade") == "CRÍTICA")
    )

    .count()
)


if testes_criticos_reprovados > 0:

    raise Exception(
        "Existem testes críticos de qualidade "
        "reprovados. A análise financeira "
        "não será processada."
    )


print(
    "Quality Gate aprovado."
)

In [0]:
df_receber = carregar_silver(
    "fato_contas_receber"
)

df_pagar = carregar_silver(
    "fato_contas_pagar"
)

df_clientes = carregar_silver(
    "dim_cliente"
)

df_fornecedores = carregar_silver(
    "dim_fornecedor"
)

In [0]:
ultima_emissao_receber = (

    df_receber

    .agg(
        F.max(
            "data_emissao"
        ).alias(
            "data"
        )
    )

    .first()["data"]
)


ultima_emissao_pagar = (

    df_pagar

    .agg(
        F.max(
            "data_emissao"
        ).alias(
            "data"
        )
    )

    .first()["data"]
)


data_referencia = max(
    ultima_emissao_receber,
    ultima_emissao_pagar
)


print(
    f"Data de referência financeira: "
    f"{data_referencia}"
)

In [0]:
df_receber_analitico = (

    df_receber

    .withColumn(

        "saldo_aberto",

        F.when(
            F.col(
                "status_titulo"
            ).isin(
                [
                    "A vencer",
                    "Vencido"
                ]
            ),

            F.col(
                "valor_titulo"
            )
        )

        .otherwise(
            F.lit(0)
        )
    )

    .withColumn(

        "valor_vencido",

        F.when(
            F.col(
                "status_titulo"
            ) == "Vencido",

            F.col(
                "valor_titulo"
            )
        )

        .otherwise(
            F.lit(0)
        )
    )

    .withColumn(

        "valor_a_vencer",

        F.when(
            F.col(
                "status_titulo"
            ) == "A vencer",

            F.col(
                "valor_titulo"
            )
        )

        .otherwise(
            F.lit(0)
        )
    )
)

In [0]:
df_receber_analitico = (

    df_receber_analitico

    .withColumn(

        "dias_atraso_referencia",

        F.when(
            (F.col("status_titulo") == "Vencido")
            &
            (
                F.col("data_vencimento")
                <=
                F.lit(data_referencia)
            ),

            F.datediff(
                F.lit(
                    data_referencia
                ),
                F.col(
                    "data_vencimento"
                )
            )
        )

        .otherwise(
            F.lit(0)
        )
    )
)

In [0]:
df_receber_analitico = (

    df_receber_analitico

    .withColumn(

        "faixa_aging",

        F.when(
            F.col(
                "status_titulo"
            ) == "A vencer",

            "A vencer"
        )

        .when(
            F.col(
                "dias_atraso_referencia"
            ).between(
                1,
                30
            ),

            "Vencido de 1 a 30 dias"
        )

        .when(
            F.col(
                "dias_atraso_referencia"
            ).between(
                31,
                60
            ),

            "Vencido de 31 a 60 dias"
        )

        .when(
            F.col(
                "dias_atraso_referencia"
            ).between(
                61,
                90
            ),

            "Vencido de 61 a 90 dias"
        )

        .when(
            F.col(
                "dias_atraso_referencia"
            ) > 90,

            "Vencido acima de 90 dias"
        )

        .otherwise(
            "Liquidado"
        )
    )
)

In [0]:
df_aging_contas_receber = (

    df_receber_analitico

    .filter(
        F.col(
            "status_titulo"
        ).isin(
            [
                "A vencer",
                "Vencido"
            ]
        )
    )

    .groupBy(
        "faixa_aging"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "quantidade_titulos"
        ),

        F.round(
            F.sum(
                "saldo_aberto"
            ),
            2
        ).alias(
            "valor_carteira"
        ),

        F.countDistinct(
            "id_cliente"
        ).alias(
            "quantidade_clientes"
        )
    )
)

In [0]:
valor_total_carteira = (

    df_aging_contas_receber

    .agg(
        F.sum(
            "valor_carteira"
        ).alias(
            "total"
        )
    )

    .first()["total"]
)

In [0]:
df_aging_contas_receber = (

    df_aging_contas_receber

    .withColumn(

        "participacao_carteira_percentual",

        F.when(
            F.lit(
                valor_total_carteira
            ) > 0,

            F.round(
                F.col(
                    "valor_carteira"
                )
                /
                F.lit(
                    valor_total_carteira
                )
                * 100,
                2
            )
        )

        .otherwise(
            F.lit(0)
        )
    )

    .withColumn(
        "data_referencia",
        F.lit(
            data_referencia
        )
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
salvar_gold(
    df_aging_contas_receber,
    "aging_contas_receber"
)

In [0]:
display(
    df_aging_contas_receber
)

In [0]:
df_inadimplencia_clientes = (

    df_receber_analitico

    .groupBy(
        "id_cliente"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "quantidade_titulos"
        ),

        F.round(
            F.sum(
                "valor_titulo"
            ),
            2
        ).alias(
            "valor_total_titulos"
        ),

        F.round(
            F.sum(
                "saldo_aberto"
            ),
            2
        ).alias(
            "saldo_aberto"
        ),

        F.round(
            F.sum(
                "valor_vencido"
            ),
            2
        ).alias(
            "valor_vencido"
        ),

        F.max(
            "dias_atraso_referencia"
        ).alias(
            "maior_atraso_dias"
        ),

        F.sum(
            F.when(
                F.col(
                    "status_titulo"
                ) == "Vencido",
                1
            ).otherwise(0)
        ).alias(
            "quantidade_titulos_vencidos"
        ),

        F.sum(
            F.when(
                F.col(
                    "status_titulo"
                ) == "Pago em atraso",
                1
            ).otherwise(0)
        ).alias(
            "titulos_pagos_em_atraso"
        )
    )
)

In [0]:
df_inadimplencia_clientes = (

    df_receber_analitico

    .groupBy(
        "id_cliente"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "quantidade_titulos"
        ),

        F.round(
            F.sum(
                "valor_titulo"
            ),
            2
        ).alias(
            "valor_total_titulos"
        ),

        F.round(
            F.sum(
                "saldo_aberto"
            ),
            2
        ).alias(
            "saldo_aberto"
        ),

        F.round(
            F.sum(
                "valor_vencido"
            ),
            2
        ).alias(
            "valor_vencido"
        ),

        F.max(
            "dias_atraso_referencia"
        ).alias(
            "maior_atraso_dias"
        ),

        F.sum(
            F.when(
                F.col(
                    "status_titulo"
                ) == "Vencido",
                1
            ).otherwise(0)
        ).alias(
            "quantidade_titulos_vencidos"
        ),

        F.sum(
            F.when(
                F.col(
                    "status_titulo"
                ) == "Pago em atraso",
                1
            ).otherwise(0)
        ).alias(
            "titulos_pagos_em_atraso"
        )
    )
)

In [0]:
df_inadimplencia_clientes = (

    df_inadimplencia_clientes.alias("f")

    .join(
        df_clientes.alias("c"),
        on="id_cliente",
        how="left"
    )

    .select(

        "id_cliente",

        F.col(
            "c.nome_cliente"
        ).alias(
            "nome_cliente"
        ),

        F.col(
            "c.segmento_cliente"
        ).alias(
            "segmento_cliente"
        ),

        F.col(
            "c.porte_cliente"
        ).alias(
            "porte_cliente"
        ),

        F.col(
            "c.cidade"
        ).alias(
            "cidade"
        ),

        F.col(
            "f.quantidade_titulos"
        ),

        F.col(
            "f.valor_total_titulos"
        ),

        F.col(
            "f.saldo_aberto"
        ),

        F.col(
            "f.valor_vencido"
        ),

        F.col(
            "f.maior_atraso_dias"
        ),

        F.col(
            "f.quantidade_titulos_vencidos"
        ),

        F.col(
            "f.titulos_pagos_em_atraso"
        )
    )
)

In [0]:
df_inadimplencia_clientes = (

    df_inadimplencia_clientes

    .withColumn(

        "percentual_saldo_vencido",

        F.when(
            F.col(
                "saldo_aberto"
            ) > 0,

            F.round(
                F.col(
                    "valor_vencido"
                )
                /
                F.col(
                    "saldo_aberto"
                )
                * 100,
                2
            )
        )

        .otherwise(
            F.lit(0)
        )
    )
)

In [0]:
df_inadimplencia_clientes = (

    df_inadimplencia_clientes

    .withColumn(

        "nivel_atencao_financeira",

        F.when(
            F.col(
                "maior_atraso_dias"
            ) > 90,

            "Crítico"
        )

        .when(
            F.col(
                "maior_atraso_dias"
            ) > 60,

            "Alto"
        )

        .when(
            F.col(
                "maior_atraso_dias"
            ) > 30,

            "Médio"
        )

        .when(
            F.col(
                "maior_atraso_dias"
            ) > 0,

            "Atenção"
        )

        .otherwise(
            "Regular"
        )
    )

    .withColumn(
        "data_referencia",
        F.lit(
            data_referencia
        )
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
salvar_gold(
    df_inadimplencia_clientes,
    "inadimplencia_clientes"
)

In [0]:
display(

    df_inadimplencia_clientes

    .filter(
        F.col(
            "valor_vencido"
        ) > 0
    )

    .select(
        "nome_cliente",
        "segmento_cliente",
        "porte_cliente",
        "saldo_aberto",
        "valor_vencido",
        "maior_atraso_dias",
        "quantidade_titulos_vencidos",
        "nivel_atencao_financeira"
    )

    .orderBy(
        F.desc(
            "valor_vencido"
        )
    )

    .limit(30)
)

In [0]:
df_pagar_analitico = (

    df_pagar

    .withColumn(

        "saldo_aberto",

        F.when(
            F.col(
                "status_titulo"
            ).isin(
                [
                    "A vencer",
                    "Vencido"
                ]
            ),

            F.col(
                "valor_titulo"
            )
        )

        .otherwise(
            F.lit(0)
        )
    )

    .withColumn(

        "dias_atraso_referencia",

        F.when(
            F.col(
                "status_titulo"
            ) == "Vencido",

            F.datediff(
                F.lit(
                    data_referencia
                ),
                F.col(
                    "data_vencimento"
                )
            )
        )

        .otherwise(
            F.lit(0)
        )
    )
)

In [0]:
df_pagar_analitico = (

    df_pagar_analitico

    .withColumn(

        "faixa_aging",

        F.when(
            F.col(
                "status_titulo"
            ) == "A vencer",

            "A vencer"
        )

        .when(
            F.col(
                "dias_atraso_referencia"
            ).between(
                1,
                30
            ),

            "Vencido de 1 a 30 dias"
        )

        .when(
            F.col(
                "dias_atraso_referencia"
            ).between(
                31,
                60
            ),

            "Vencido de 31 a 60 dias"
        )

        .when(
            F.col(
                "dias_atraso_referencia"
            ).between(
                61,
                90
            ),

            "Vencido de 61 a 90 dias"
        )

        .when(
            F.col(
                "dias_atraso_referencia"
            ) > 90,

            "Vencido acima de 90 dias"
        )

        .otherwise(
            "Liquidado"
        )
    )
)

In [0]:
df_aging_contas_pagar = (

    df_pagar_analitico

    .filter(
        F.col(
            "status_titulo"
        ).isin(
            [
                "A vencer",
                "Vencido"
            ]
        )
    )

    .groupBy(
        "faixa_aging"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "quantidade_titulos"
        ),

        F.round(
            F.sum(
                "saldo_aberto"
            ),
            2
        ).alias(
            "valor_carteira"
        ),

        F.countDistinct(
            "id_fornecedor"
        ).alias(
            "quantidade_fornecedores"
        )
    )

    .withColumn(
        "data_referencia",
        F.lit(
            data_referencia
        )
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
salvar_gold(
    df_aging_contas_pagar,
    "aging_contas_pagar"
)

In [0]:
prazo_medio_recebimento = (

    df_receber

    .filter(
        F.col(
            "data_pagamento"
        ).isNotNull()
    )

    .withColumn(

        "prazo_recebimento",

        F.datediff(
            F.col(
                "data_pagamento"
            ),
            F.col(
                "data_emissao"
            )
        )
    )

    .agg(
        F.round(
            F.avg(
                "prazo_recebimento"
            ),
            2
        ).alias(
            "prazo_medio"
        )
    )

    .first()[
        "prazo_medio"
    ]
)

In [0]:
kpis_receber = (

    df_receber_analitico

    .agg(

        F.round(
            F.sum(
                "saldo_aberto"
            ),
            2
        ).alias(
            "total_receber_aberto"
        ),

        F.round(
            F.sum(
                "valor_a_vencer"
            ),
            2
        ).alias(
            "total_a_vencer"
        ),

        F.round(
            F.sum(
                "valor_vencido"
            ),
            2
        ).alias(
            "total_vencido"
        )
    )

    .first()
)

In [0]:
kpis_pagar = (

    df_pagar_analitico

    .agg(

        F.round(
            F.sum(
                "saldo_aberto"
            ),
            2
        ).alias(
            "total_pagar_aberto"
        ),

        F.round(
            F.sum(
                F.when(
                    F.col(
                        "status_titulo"
                    ) == "Vencido",

                    F.col(
                        "valor_titulo"
                    )
                )

                .otherwise(
                    F.lit(0)
                )
            ),
            2
        ).alias(
            "total_pagar_vencido"
        )
    )

    .first()
)

In [0]:
total_receber_aberto = float(
    kpis_receber[
        "total_receber_aberto"
    ]
)

total_vencido = float(
    kpis_receber[
        "total_vencido"
    ]
)


taxa_inadimplencia = (

    (
        total_vencido
        /
        total_receber_aberto
        * 100
    )

    if total_receber_aberto > 0

    else 0
)

In [0]:
data_limite_30_dias = (
    data_referencia
    + timedelta(days=30)
)

In [0]:
entradas_previstas_30d = (

    df_receber

    .filter(
        (F.col("status_titulo") == "A vencer")
        &
        (
            F.col("data_vencimento")
            >
            F.lit(
                data_referencia
            )
        )
        &
        (
            F.col("data_vencimento")
            <=
            F.lit(
                data_limite_30_dias
            )
        )
    )

    .agg(
        F.coalesce(
            F.sum(
                "valor_titulo"
            ),
            F.lit(0)
        ).alias(
            "total"
        )
    )

    .first()["total"]
)

In [0]:
saidas_previstas_30d = (

    df_pagar

    .filter(
        (F.col("status_titulo") == "A vencer")
        &
        (
            F.col("data_vencimento")
            >
            F.lit(
                data_referencia
            )
        )
        &
        (
            F.col("data_vencimento")
            <=
            F.lit(
                data_limite_30_dias
            )
        )
    )

    .agg(
        F.coalesce(
            F.sum(
                "valor_titulo"
            ),
            F.lit(0)
        ).alias(
            "total"
        )
    )

    .first()["total"]
)

In [0]:
resumo_financeiro = [
    (
        data_referencia,

        float(
            kpis_receber[
                "total_receber_aberto"
            ]
        ),

        float(
            kpis_receber[
                "total_a_vencer"
            ]
        ),

        float(
            kpis_receber[
                "total_vencido"
            ]
        ),

        round(
            taxa_inadimplencia,
            2
        ),

        float(
            prazo_medio_recebimento
        ),

        float(
            kpis_pagar[
                "total_pagar_aberto"
            ]
        ),

        float(
            kpis_pagar[
                "total_pagar_vencido"
            ]
        ),

        float(
            entradas_previstas_30d
        ),

        float(
            saidas_previstas_30d
        ),

        round(
            float(
                entradas_previstas_30d
            )
            -
            float(
                saidas_previstas_30d
            ),
            2
        )
    )
]

In [0]:
df_resumo_financeiro = (

    spark.createDataFrame(

        resumo_financeiro,

        [
            "data_referencia",
            "total_receber_aberto",
            "total_a_vencer",
            "total_receber_vencido",
            "taxa_inadimplencia_percentual",
            "prazo_medio_recebimento_dias",
            "total_pagar_aberto",
            "total_pagar_vencido",
            "entradas_previstas_30d",
            "saidas_previstas_30d",
            "saldo_previsto_30d"
        ]
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
salvar_gold(
    df_resumo_financeiro,
    "resumo_financeiro_atual"
)

In [0]:
display(
    df_resumo_financeiro
)

In [0]:
df_recebimentos_realizados = (

    df_receber

    .filter(
        F.col(
            "data_pagamento"
        ).isNotNull()
    )

    .withColumn(
        "mes_referencia",
        F.trunc(
            "data_pagamento",
            "month"
        )
    )

    .groupBy(
        "mes_referencia"
    )

    .agg(
        F.round(
            F.sum(
                "valor_titulo"
            ),
            2
        ).alias(
            "entradas"
        )
    )

    .withColumn(
        "saidas",
        F.lit(0.0)
    )

    .withColumn(
        "tipo_fluxo",
        F.lit(
            "Realizado"
        )
    )
)

In [0]:
df_pagamentos_realizados = (

    df_pagar

    .filter(
        F.col(
            "data_pagamento"
        ).isNotNull()
    )

    .withColumn(
        "mes_referencia",
        F.trunc(
            "data_pagamento",
            "month"
        )
    )

    .groupBy(
        "mes_referencia"
    )

    .agg(
        F.round(
            F.sum(
                "valor_titulo"
            ),
            2
        ).alias(
            "saidas"
        )
    )

    .withColumn(
        "entradas",
        F.lit(0.0)
    )

    .withColumn(
        "tipo_fluxo",
        F.lit(
            "Realizado"
        )
    )
)

In [0]:
df_fluxo_realizado = (

    df_recebimentos_realizados

    .unionByName(
        df_pagamentos_realizados
    )

    .groupBy(
        "mes_referencia",
        "tipo_fluxo"
    )

    .agg(

        F.round(
            F.sum(
                "entradas"
            ),
            2
        ).alias(
            "entradas"
        ),

        F.round(
            F.sum(
                "saidas"
            ),
            2
        ).alias(
            "saidas"
        )
    )
)

In [0]:
df_recebimentos_projetados = (

    df_receber

    .filter(
        F.col(
            "status_titulo"
        ).isin(
            [
                "A vencer",
                "Vencido"
            ]
        )
    )

    .withColumn(

        "data_fluxo",

        F.greatest(
            F.col(
                "data_vencimento"
            ),
            F.lit(
                data_referencia
            )
        )
    )

    .withColumn(
        "mes_referencia",
        F.trunc(
            "data_fluxo",
            "month"
        )
    )

    .groupBy(
        "mes_referencia"
    )

    .agg(
        F.round(
            F.sum(
                "valor_titulo"
            ),
            2
        ).alias(
            "entradas"
        )
    )

    .withColumn(
        "saidas",
        F.lit(0.0)
    )

    .withColumn(
        "tipo_fluxo",
        F.lit(
            "Projetado"
        )
    )
)

In [0]:
df_pagamentos_projetados = (

    df_pagar

    .filter(
        F.col(
            "status_titulo"
        ).isin(
            [
                "A vencer",
                "Vencido"
            ]
        )
    )

    .withColumn(

        "data_fluxo",

        F.greatest(
            F.col(
                "data_vencimento"
            ),
            F.lit(
                data_referencia
            )
        )
    )

    .withColumn(
        "mes_referencia",
        F.trunc(
            "data_fluxo",
            "month"
        )
    )

    .groupBy(
        "mes_referencia"
    )

    .agg(
        F.round(
            F.sum(
                "valor_titulo"
            ),
            2
        ).alias(
            "saidas"
        )
    )

    .withColumn(
        "entradas",
        F.lit(0.0)
    )

    .withColumn(
        "tipo_fluxo",
        F.lit(
            "Projetado"
        )
    )
)

In [0]:
df_fluxo_projetado = (

    df_recebimentos_projetados

    .unionByName(
        df_pagamentos_projetados
    )

    .groupBy(
        "mes_referencia",
        "tipo_fluxo"
    )

    .agg(

        F.round(
            F.sum(
                "entradas"
            ),
            2
        ).alias(
            "entradas"
        ),

        F.round(
            F.sum(
                "saidas"
            ),
            2
        ).alias(
            "saidas"
        )
    )
)

In [0]:
df_fluxo_caixa_mensal = (

    df_fluxo_realizado

    .unionByName(
        df_fluxo_projetado
    )

    .withColumn(

        "saldo_periodo",

        F.round(
            F.col(
                "entradas"
            )
            -
            F.col(
                "saidas"
            ),
            2
        )
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
salvar_gold(
    df_fluxo_caixa_mensal,
    "fluxo_caixa_mensal"
)

In [0]:
display(

    df_fluxo_caixa_mensal

    .orderBy(
        "mes_referencia",
        "tipo_fluxo"
    )
)

In [0]:
valor_vencido_clientes = (

    df_inadimplencia_clientes

    .agg(
        F.round(
            F.sum(
                "valor_vencido"
            ),
            2
        ).alias(
            "total"
        )
    )

    .first()["total"]
)


valor_vencido_resumo = (
    kpis_receber[
        "total_vencido"
    ]
)


diferenca = abs(
    float(
        valor_vencido_clientes
    )
    -
    float(
        valor_vencido_resumo
    )
)


if diferenca > 0.05:

    raise Exception(
        "O valor vencido por cliente "
        "não corresponde ao resumo financeiro."
    )


print(
    "Validação concluída: "
    "inadimplência consistente."
)

In [0]:
valor_aging = (

    df_aging_contas_receber

    .agg(
        F.round(
            F.sum(
                "valor_carteira"
            ),
            2
        ).alias(
            "total"
        )
    )

    .first()["total"]
)


diferenca_aging = abs(
    float(valor_aging)
    -
    float(total_receber_aberto)
)


if diferenca_aging > 0.05:

    raise Exception(
        "O Aging não corresponde "
        "à carteira em aberto."
    )


print(
    "Validação concluída: "
    "Aging consistente com a carteira aberta."
)

In [0]:
display(

    df_inadimplencia_clientes

    .filter(
        F.col(
            "valor_vencido"
        ) > 0
    )

    .select(
        "nome_cliente",
        "segmento_cliente",
        "porte_cliente",
        "valor_vencido",
        "maior_atraso_dias",
        "nivel_atencao_financeira"
    )

    .orderBy(
        F.desc(
            "valor_vencido"
        )
    )

    .limit(20)
)

In [0]:
display(

    df_aging_contas_receber

    .filter(
        F.col(
            "faixa_aging"
        )
        ==
        "Vencido acima de 90 dias"
    )
)

In [0]:
display(

    df_fluxo_caixa_mensal

    .filter(
        F.col(
            "tipo_fluxo"
        )
        ==
        "Projetado"
    )

    .orderBy(
        "mes_referencia"
    )
)

In [0]:
comentarios = {

    "resumo_financeiro_atual":
        "Principais indicadores atuais de contas a receber, contas a pagar, inadimplência e projeção financeira.",

    "aging_contas_receber":
        "Distribuição da carteira de contas a receber em aberto por faixa de vencimento.",

    "aging_contas_pagar":
        "Distribuição das obrigações financeiras em aberto por faixa de vencimento.",

    "inadimplencia_clientes":
        "Indicadores de inadimplência e exposição financeira consolidados por cliente.",

    "fluxo_caixa_mensal":
        "Fluxo mensal de entradas, saídas e saldo financeiro realizado e projetado."
}


for tabela, comentario in comentarios.items():

    spark.sql(
        f"""
        COMMENT ON TABLE
        `{catalogo_atual}`.`{schema_gold}`.`{tabela}`
        IS '{comentario}'
        """
    )


print(
    "Descrições adicionadas às tabelas financeiras."
)